In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.stable_cp import StableConformalPredictor

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    lam=0.5,
    kernel="laplacian",
    solver="lbfgs",
    loss_name=loss_name,
    loss_params=loss_params,
)

Instantiate region predictor

In [7]:
conformal_predictor = StableConformalPredictor(
    predictor, non_conformity_name="absolute"
)
region_predictor = conformal_predictor.fit_predict(
    train_input_points, train_output_points, test_input_points
)

In [9]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

[0.00504479]
[0.00644728]
[0.00728995]
[0.00686468]
[0.00740733]
[0.00643532]
[0.00731954]
[0.00681265]


KeyboardInterrupt: 

In [ ]:
prediction_regions

[{'upper': [-1.6243772842524116,1.6409788428973695],
  'lower': [-1.6154001383833214,1.6318260265308369]},
 {'upper': [-1.6294023478593496,1.635960131824179],
  'lower': [-1.620372058706724,1.6268604490569936]},
 {'upper': [-1.6257352397944602,1.6396233509620675],
  'lower': [-1.6167437365328465,1.6304848882325573]},
 {'upper': [-1.626985222766037,1.6383726159721574],
  'lower': [-1.6179804896540104,1.629247384237708]},
 {'upper': [-1.6279266740664662,1.6374328351977938],
  'lower': [-1.6189119865535644,1.6283175553174007]},
 {'upper': [-1.6280252873089838,1.6373348018378848],
  'lower': [-1.6190095589397309,1.6282205619305288]},
 {'upper': [-1.6237139729485894,1.6416402571393716],
  'lower': [-1.6147438368283962,1.632480433915504]},
 {'upper': [-1.6252964101605347,1.640061027013053],
  'lower': [-1.6163095449842517,1.630917927954687]},
 {'upper': [-1.6263433037792072,1.6390154336056677],
  'lower': [-1.6173453671125,1.6298834040565393]},
 {'upper': [-1.627516100551675,1.63784358717869

In [11]:
coverage_upper = np.mean(
    [
        test_output_point in prediction_region["upper"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_upper)

test coverage:  0.84


In [12]:
coverage_lower = np.mean(
    [
        test_output_point in prediction_region["lower"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_lower)

test coverage:  0.84
